## Path configuration

In [67]:
from pathlib import Path
import os

PROJECT_NAME = "MALDIAlign"

cwd = Path().resolve()

# Walk upwards until we find the project folder
target = None
for parent in [cwd] + list(cwd.parents):
    if parent.name == PROJECT_NAME:
        target = parent
        break

# If the project folder is found and we are not already there, then change cwd
if target is not None and target != cwd:
    os.chdir(target)

print("Working directory:", os.getcwd())

Working directory: /export/usuarios01/agnavarr/MALDIAlign


## Imports

In [68]:
import numpy as np
import pandas as pd

# utils
from utils.config import load_config
from utils.data import load_pkl
from utils.metrics import *
from utils.viz import *
from utils.style_transfer import style_transfer

# tools
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# models
from sklearn.linear_model import LogisticRegression

## Data loading

In [69]:
cfg = load_config()
driams_pkl = cfg["data"]["DRIAMS_REDUCED_PKL"]

In [70]:
driams = load_pkl(driams_pkl)

In [71]:
data, label, meta = driams["data"], driams["label"], pd.DataFrame.from_records(list(driams["meta"]))

In [72]:
# Apply normalization: scale each spectrum to [0, 1]
X_min = data.min(axis=1, keepdims=True)
X_max = data.max(axis=1, keepdims=True)
data_norm = (data - X_min) / (X_max - X_min + 1e-8)

In [73]:
# Filter the data by hospital
filtered_data = {}
for hosp in meta["hospital"].unique():
    idx = np.where(meta["hospital"].values == hosp)[0]
    filtered_data[hosp] = {
        "data": data_norm[idx],
        "label": label[idx],
        "meta": meta.iloc[idx]
    }

In [74]:
# Declare the datasets
dataA, labelA, metaA = filtered_data["DRIAMS_A"]["data"], filtered_data["DRIAMS_A"]["label"], filtered_data["DRIAMS_A"]["meta"]
dataD, labelD, metaD = filtered_data["DRIAMS_D"]["data"], filtered_data["DRIAMS_D"]["label"], filtered_data["DRIAMS_D"]["meta"]

## Style transfer

#### Utils

In [75]:
import torch
from models.deep.MultiVAEPrior import MultiVAE_Bernoulli_SpeciesPrior_Extended

In [76]:
def load_model(model, path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    return model

def encode_latent(model, X, device, batch_size=256):
    model.eval()
    Z = []

    X_tensor = torch.tensor(X, dtype=torch.float32)

    loader = torch.utils.data.DataLoader(
        X_tensor,
        batch_size=batch_size,
        shuffle=False
    )

    with torch.no_grad():
        for x in loader:
            x = x.to(device)
            mu, _ = model.encoder(x)
            Z.append(mu.cpu().numpy())

    return np.vstack(Z)

#### Load MultiDecoder

In [16]:
meta_all = pd.concat([metaA, metaD], ignore_index=True)
data_all = np.vstack([dataA, dataD])
label_all = np.concatenate([labelA,labelD])

In [15]:
vae_path = "experiments/results/vae_multidecoder_prior/20260116_084734/model.pth"

backbone = MultiVAE_Bernoulli_SpeciesPrior_Extended(
    input_dim=dataA.shape[1],
    latent_dim=64,
    num_domains=2,
    n_species=len(np.unique(labelA))
)

vae = load_model(backbone, vae_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Train Logistic Regression to classify domains

In [77]:
X = data_all
y = meta_all["hospital"].values

# Train / Test (hold-out para style transfer)
X_tr, X_test, y_tr, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Train / Val (opcional, para sanity check)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr,
    test_size=0.4,
    stratify=y_tr,
    random_state=42
)

In [78]:
clf_x = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

clf_x.fit(X_tr, y_tr)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


In [79]:
val_ba = balanced_accuracy_score(y_val, clf_x.predict(X_val))
print("Domain BA on validation (x):", val_ba)

Domain BA on validation (x): 0.9844793879336031


#### Transfer A to D

In [99]:
DOMAIN_MAP = {
    "DRIAMS_A": 0,
    "DRIAMS_D": 1,
}

In [100]:
X_testA = X_test[np.where(y_test == "DRIAMS_A")]

In [101]:
X_A_to_D = style_transfer(
    model=vae,
    X=X_testA,
    domain_target_id=DOMAIN_MAP["DRIAMS_D"],
    device=device
)

In [102]:
y_pred = clf_x.predict(X_A_to_D)

pred_percent = pd.Series(y_pred).value_counts(normalize=True) * 100
print(pred_percent)

DRIAMS_D    97.626753
DRIAMS_A     2.373247
Name: proportion, dtype: float64


#### Transfer D to A

In [96]:
X_testD = X_test[np.where(y_test == "DRIAMS_D")]

In [97]:
X_D_to_A = style_transfer(
    model=vae,
    X=X_testD,
    domain_target_id=DOMAIN_MAP["DRIAMS_A"],
    device=device
)

In [98]:
y_pred2 = clf_x.predict(X_D_to_A)
pred_percent2 = pd.Series(y_pred2).value_counts(normalize=True) * 100
print(pred_percent2)

DRIAMS_A    97.127223
DRIAMS_D     2.872777
Name: proportion, dtype: float64
